# Meteostat station-record missingness: what can observations really tell us?

**Optional companion notebook — not the primary climate trend project.**

The weather-station series previously examined for Dar es Salaam is too incomplete for a reliable direct comparison of **1986–1995** with **2016–2025** under our coverage criterion. This notebook explores *why* missing observations matter rather than trying to create a result by relaxing the rules.

Meteostat station records are different from the [NASA POWER gridded reanalysis](east_africa_temperature_trends.ipynb). Neither dataset is guaranteed to be free from biases. The exercise only runs on actual downloaded station observations; it never substitutes artificial values for station measurements.

Download first (while online):

```sh
uv run python tools/download_meteostat.py --station 63894
```

We intentionally disable model-derived fills in the Meteostat downloader. Keep actual observations private until their redistribution terms are confirmed. See [project guidance](../projects/meteostat-temperature/README.md).

## 1. What observations do we have?

Explore the days and months with observations. Do not interpolate, use `.ffill()`, or treat absent days as a new measurement. Missingness may be non-random, particularly in earlier decades.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from tools.temperature_methods import annual_means

STATION = ROOT / "data/private/meteostat_63894_1986_2025.csv"
station = None
if STATION.exists():
    station = pd.read_csv(STATION, parse_dates=["date"]).set_index("date")
    if station.index.has_duplicates:
        raise ValueError("Duplicate dates in downloaded station observations")
    station = pd.to_numeric(station["temp"], errors="coerce").asfreq("D").loc["1986":"2025"]
    print("Meteostat station archive: observed days", int(station.notna().sum()), "/", len(station))
else:
    print("No Meteostat CSV. Download while online; station analysis is skipped.")
    print("Command: uv run python tools/download_meteostat.py --station 63894")

## 2. Plot annual and monthly coverage

The 20-days-per-month completeness rule was selected *before* inspecting the trend. What would happen if we ignored the missing months and computed yearly averages anyway?

In [ ]:
if station is not None:
    coverage = station.notna().groupby(station.index.year).mean()
    monthly_counts = station.notna().resample("MS").sum()
    fig, axes = plt.subplots(2, 1, figsize=(11, 6))
    coverage.plot(ax=axes[0], marker="o", title="Meteostat: fraction of observed days per year")
    axes[0].set_ylim(0, 1.05)
    monthly_counts.plot(ax=axes[1], title="Meteostat: number of observed days per month")
    axes[1].set_ylabel("Observed days")
    plt.tight_layout()
    plt.show()
    annual, monthly = annual_means(station, min_days_per_month=20)
    print("Eligible years:", list(annual.index))
    print("Early decade (1986–1995):", len(annual.loc[1986:1995]), "/ 10")
    print("Recent decade (2016–2025):", len(annual.loc[2016:2025]), "/ 10")
else:
    print("Coverage diagnostics require the station CSV.")

## 3. A cautious comparison with NASA POWER

If both datasets have been downloaded, compare **only days for which Meteostat has an actual temperature**. NASA POWER estimates a coarse grid-cell mean, while Meteostat reports a station-level record; their differences reflect spatial scale, model error, station-specific effects and observation gaps.

The comparison is **not** an independent validation of every historical NASA POWER value, especially if the reanalysis assimilates related observations. Does coverage change which seasons dominate the overlap?

In [ ]:
POWER = ROOT / "data/private/nasa_power_dar_es_salaam_1986_2025.csv"
if station is not None and POWER.exists():
    gridded = pd.read_csv(POWER, parse_dates=["date"]).set_index("date")["temp"]
    overlap = pd.concat(
        [station.rename("station"), gridded.rename("power_grid_cell")],
        axis=1, join="inner"
    ).dropna()
    print("Days with both station and POWER estimates:", len(overlap))
    if len(overlap):
        diff = overlap["station"] - overlap["power_grid_cell"]
        print("Station minus grid-cell mean on overlapping days:", round(float(diff.mean()), 3), "°C")
        overlap.plot.scatter(x="power_grid_cell", y="station", s=3, alpha=0.2,
                             title="Only overlapping days; different quantities")
        plt.show()
        seasonal_count = diff.groupby(diff.index.month).size()
        print("Overlapping days by calendar month:", seasonal_count.to_dict())
else:
    print("Download both sources to compare matched dates. No fake values will be substituted.")

## Discussion for the group

1. Why is the station dataset not suitable for the original 1986–2025 trend question?
2. Can a nearly complete reanalysis help us form a research question without being a substitute for measured station history?
3. What station metadata or information from TMA would strengthen the investigation?
4. What if our only overlap occurs during the more recent years — does the plot tell us anything about 1986?
5. What conclusions are supported, and which are **not**?

An honest report on insufficient data is useful scientific work. Never present missing observations as measurements or extend conclusions beyond available evidence.